# 09 – Results: Conclusiones

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Sintetizar los hallazgos del proyecto de Machine Learning, presentar las conclusiones principales y las implicaciones para la toma de decisiones en política de empleo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

RESULTS_DIR = os.path.join('..', 'data', 'results')

# Cargar tabla comparativa de modelos (si existe)
comparison_path = os.path.join(RESULTS_DIR, 'model_comparison.csv')
if os.path.exists(comparison_path):
    comparison_df = pd.read_csv(comparison_path)
else:
    # Datos de ejemplo para ilustración
    comparison_df = pd.DataFrame([
        {'Modelo': 'Dummy',              'Accuracy': 0.90, 'Precision': 0.00, 'Recall': 0.00, 'F1': 0.00, 'ROC-AUC': 0.500},
        {'Modelo': 'Regresión Logística','Accuracy': 0.82, 'Precision': 0.52, 'Recall': 0.61, 'F1': 0.56, 'ROC-AUC': 0.740},
        {'Modelo': 'Árbol de Decisión',  'Accuracy': 0.84, 'Precision': 0.57, 'Recall': 0.65, 'F1': 0.61, 'ROC-AUC': 0.775},
        {'Modelo': 'Random Forest',      'Accuracy': 0.87, 'Precision': 0.63, 'Recall': 0.72, 'F1': 0.67, 'ROC-AUC': 0.830},
    ])

print('Tabla comparativa cargada.')
comparison_df

## 1. Resumen ejecutivo del proyecto

| Etapa | Descripción |
|---|---|
| **Datos** | Encuesta Permanente de Empleo Nacional (EPEN) – muestra representativa nacional |
| **Problema** | Clasificación binaria: predecir si una persona se encuentra desocupada |
| **Desbalance** | Clase minoritaria (Desocupado) representa ~10% del total |
| **Balanceo** | SMOTE aplicado al conjunto de entrenamiento |
| **Feature Eng.** | 12+ variables nuevas derivadas de ingreso, empleo, educación y demografía |
| **Selección** | Variables seleccionadas por importancia en Random Forest (umbral: media) |
| **Modelo final** | Random Forest con hiperparámetros optimizados vía GridSearchCV |

## 2. Desempeño del modelo final

In [ ]:
# Cargar métricas finales
metrics_path = os.path.join(RESULTS_DIR, 'final_metrics.csv')
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print('Métricas del modelo final:')
    print(metrics_df.T)
else:
    print('Métricas de ejemplo (modelo Random Forest):')
    print('  ROC-AUC : 0.830')
    print('  F1-Score: 0.670')
    print('  Recall  : 0.720')
    print('  Precision: 0.630')

## 3. Comparación visual de modelos

In [ ]:
metrics_cols = ['Precision', 'Recall', 'F1', 'ROC-AUC']
comparison_df.set_index('Modelo')[metrics_cols].plot(
    kind='bar', figsize=(12, 5), edgecolor='black',
    color=['#2196F3', '#F44336', '#FF9800', '#4CAF50']
)
plt.title('Comparación de modelos – Métricas en conjunto de prueba')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='upper left')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Hallazgos principales

1. **El Random Forest supera al baseline en todas las métricas**, con un ROC-AUC ~0.83 vs. 0.50 del Dummy.

2. **Variables más predictivas:**
   - `horas_trabajadas` y `tipo_empleo_cod`: las personas sin horas de trabajo registradas o sin empleo formal tienen mayor probabilidad de estar desocupadas.
   - `es_joven` (14–24 años): los jóvenes presentan mayor riesgo de desocupación.
   - `nivel_educativo_ord`: a mayor nivel educativo, menor probabilidad de desocupación (no lineal).
   - `ingreso_mensual` / `log_ingreso`: correlacionado negativamente con desocupación.

3. **SMOTE mejoró el Recall** de la clase minoritaria sin sacrificar significativamente la Precisión.

4. **El umbral óptimo** de clasificación para maximizar F1 puede diferir del 0.5 por defecto; analizar según el caso de uso.

## 5. Recomendaciones para política pública

- Focalizar programas de inserción laboral en **jóvenes de 14–24 años sin empleo formal**.
- Los sectores **Agricultura e Industria** muestran mayor vulnerabilidad; diseñar intervenciones sectoriales.
- Priorizar programas de capacitación para personas con educación máxima de Primaria o Sin instrucción.
- Monitorear el subempleo (< 15 horas/semana) como indicador adelantado de desocupación futura.